In [1]:
import requests
import pandas as pd
import os
from io import StringIO
from datetime import datetime

tokenCanvas = os.environ['tokenCanvas']
idag = datetime.today().isoformat()[0:10]

headers = {
    'Authorization': f'Bearer {tokenCanvas}',
}
params = {
    'per_page': 20,
    'type[]': 'StudentEnrollment',
    'state[]': 'active',
}

# Liste alle lærarar og studentar
I tilfelle krise: eg treng ei oppdatert liste over alle aktive lærarar og studentar i emne i Canvas.

Eg henter dei ved å sjekke to lister som vert sendt frå FS til Canvas kvar natt. 

1. "enrollments.csv"
Denne lister alle personar som har ei definert rolle i alle emne. Den viser sis-id.

2. "users_filtered.csv" 
Denne kobler sis_id og e-post for alle personar i FS.

In [2]:
url = "https://hvl.instructure.com/api/v1/accounts/1/sis_imports"
svar = requests.get(url, headers=headers, params=params)
importer = svar.json()
df_liste = []
for i in importer['sis_imports']:
    if ("csv_attachments" in i) and (i['created_at'] > idag):
        # print(i['created_at'])
        for a in i["csv_attachments"]:
            if a['filename'] == 'enrollments.csv':
                fil = a['url']
        data = pd.read_csv(StringIO(requests.get(fil, headers=headers).content.decode("utf-8")), dtype=str, low_memory=False)
        df_liste.append(data)
innmeldingar = pd.concat(df_liste)

In [3]:
url = "https://hvl.instructure.com/api/v1/accounts/1/sis_imports"
svar = requests.get(url, headers=headers, params=params)
importer = svar.json()
df_liste = []
for i in importer['sis_imports']:
    if ("csv_attachments" in i) and (i['created_at'] > idag):
        # print(i['created_at'])
        for a in i["csv_attachments"]:
            if a['filename'] == 'users_filtered.csv':
                fil = a['url']
        data = pd.read_csv(StringIO(requests.get(fil, headers=headers).content.decode("utf-8")), dtype=str, low_memory=False)
        df_liste.append(data)
adresser = pd.concat(df_liste)

Så kan eg plukke ut alle lærarar og studentar (og filtrere vekk slik at eg berre har unike verdiar):

In [5]:
# lærarar = pd.DataFrame(data[data['role'] == 'teacher']['user_id'].unique(), columns=['sis_id']) #.to_csv("lærarar.csv", index=False, header=False)
studentar = pd.DataFrame(data[data['role'] == 'student']['user_id'].unique(), columns=['sis_id']) #.to_csv("studentar.csv", index=False, header=False)


KeyError: 'role'

Så kan eg prøve å hente ut alle e-postane; lærarane først:
